In [18]:
import folium
import pandas as pd
from branca.element import Element

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

In [7]:
spacex_map = folium.Map(
    location=[29.5, -95.0],
    zoom_start=5
)

spacex_map

In [8]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'

spacex_df = pd.read_csv(URL)

In [9]:
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]

launch_sites_df = spacex_df.groupby(
    ['Launch Site'], as_index=False
).first()

launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]

launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [11]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]

site_map = folium.Map(
    location=nasa_coordinate,
    zoom_start=5
)

In [12]:
for index, row in launch_sites_df.iterrows():

    coordinate = [row['Lat'], row['Long']]

    # Create a circle for each launch site
    circle = folium.Circle(
        coordinate,
        radius=1000,
        color='#000000',
        fill=True
    ).add_child(
        folium.Popup(row['Launch Site'])
    )

    # Create a text marker for each launch site
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % row['Launch Site']
        )
    )

    site_map.add_child(circle)
    site_map.add_child(marker)

In [13]:
site_map

Task 1: Findings about SpaceX launch sites

Question 1: Are all launch sites in proximity to the Equator line?

No. The launch sites are not all very close to the Equator.
- The Equator is at 0° latitude.
- SpaceX launch sites are located in the Northern Hemisphere.
- Their latitudes are approximately between 28° and 34° North.

Question 2: Are all launch sites in very close proximity to the coast?

Yes, the SpaceX launch sites are generally located near coastlines.

Explanation:
- Rockets are launched near the coast to reduce risks to populated areas.
- Coastal locations provide safer flight paths over the ocean.
- They also provide access to sea transportation and recovery operations.

Task 2: Mark successful and failed launches on the map

Now we will add colored markers to identify launch outcomes.
- Green
   Successful launch (class = 1)
- Red
   Failed launch (class = 0)

In [20]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [20]:
# Step 1: Import required libraries
import folium
import pandas as pd
from folium.plugins import MarkerCluster
from IPython.display import display

# Step 2: Download and read the dataset
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
spacex_df = pd.read_csv(URL)

# Step 3: Create marker_color column
spacex_df['marker_color'] = spacex_df['class'].apply(
    lambda x: 'green' if x == 1 else 'red'
)

# Step 4: Set NASA Johnson Space Center coordinates
nasa_coordinate = [29.559684888503615, -95.0830971930759]

# Step 5: Create a fresh map
site_map = folium.Map(
    location=nasa_coordinate,
    zoom_start=5
)

# Step 6: Create maroon circles for all launch sites (Visible ONLY at high zoom levels, min_zoom=12)
# Get unique launch sites
launch_sites_df = spacex_df.groupby(
    'Launch Site', as_index=False
).first()

# Select only the final launch site
row = launch_sites_df.iloc[-1]

coordinate = [row['Lat'], row['Long']]

# Draw one circle
folium.Circle(
    location=coordinate,
    radius=1000,
    color='#900000',
    fill=True,
    fill_color='#900000',
    fill_opacity=0.15,
    popup=row['Launch Site']
).add_to(site_map)

display(site_map)

# Step 7: Create MarkerCluster
marker_cluster = MarkerCluster()

# Step 8: Add markers for each launch record
for index, record in spacex_df.iterrows():
    coordinate = [record['Lat'], record['Long']]

    marker = folium.Marker(
        location=coordinate,
        popup=folium.Popup(
            f"Launch Site: {record['Launch Site']}<br>"
            f"Launch Result: {'Success' if record['class'] == 1 else 'Failure'}",
            max_width=250
        ),
        icon=folium.Icon(
            color=record['marker_color'],
            icon='info-sign'
        )
    )

    marker_cluster.add_child(marker)

# Step 9: Add marker cluster to map
site_map.add_child(marker_cluster)

# Step 10: Display map
display(site_map)

In [22]:
from folium.plugins import MousePosition

MousePosition(
    position='bottomright',
    separator=' | ',
    prefix='Latitude / Longitude:',
    num_digits=5
).add_to(site_map)

display(site_map)